# Sanity Check (Kalman Filter)

This notebook contains a quick sanity check to ensure the models are able to train as expected using the implementation. It is using a very small subset of the dataset to run and identify issues quickly.

In [1]:
import sys

import torch

In [2]:
# Add the `code` directory to the path.
sys.path.append("../")

import dataset
import tracker
import kflearn
import kalman
import util

In [3]:
# This code is just here so I can reload the external files in case I make
# changes to them after having already imported them.
import importlib

importlib.reload(util)
importlib.reload(kalman)
importlib.reload(kflearn)
importlib.reload(tracker)
importlib.reload(dataset);

First we just check that the simulation of the Kalman filter application actually works as expected and produces for example valid covariance matrices.

In [ ]:
model = kalman.LearnedPhysics(tracker.build_constrained_physics())
raw_data = dataset.CmuPanopticDataset(path="../data/panoptic")
data = dataset.KalmanDataset("../data/kalman/train")
cams = raw_data.get_some_cams()

with torch.inference_mode():
    for i in range(len(data)):
        print(i)
        fps, track = data[i]
        kflearn.simulate_kalman_filter(model, fps, track, cams, checks=True)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
step 1 elem 0 update

================ Covariance Matrix (torch.Size([114, 114])) ================
Max Asymmetry Error | P - P^T | : 8.13e+00
  CRITICAL: Matrix has lost symmetry
Min Eigenvalue            : 4.77e-03
Max Eigenvalue            : 2.50e+05
Negative Eigenvalue Count : 0 / 114
Condition Number (λmax/λmin) : 5.25e+07
Variance Range (Min/Max Diag): 3.03e+00 / 2.48e+05


_LinAlgError: linalg.cholesky: The factorization could not be completed because the input is not positive-definite (the leading minor of order 106 is not positive-definite).

In [24]:
# Just run a single one for testing.
with torch.inference_mode():
    fps, track = data[15]
    kflearn.simulate_kalman_filter(model, fps, track, cams, checks=True)

step 1 elem 0 update

================ Covariance Matrix (torch.Size([114, 114])) ================
Max Asymmetry Error | P - P^T | : 1.33e+01
  CRITICAL: Matrix has lost symmetry
Min Eigenvalue            : -1.50e-02
Max Eigenvalue            : 2.47e+05
Negative Eigenvalue Count : 4 / 114
  CRITICAL: Matrix is NOT Positive-Definite
Variance Range (Min/Max Diag): 3.08e-01 / 2.43e+05


_LinAlgError: linalg.cholesky: The factorization could not be completed because the input is not positive-definite (the leading minor of order 39 is not positive-definite).

We can now proceed by training the Kalman filter with the on a very small subset of the complete dataset. 

In [ ]:
model = kalman.LearnedPhysics(tracker.build_constrained_physics())

# Train the model for some epochs in the dummy dataset.
kflearn.train_epochs_in(100, None, None, model, testing=True)

==== iter ====


_LinAlgError: linalg.cholesky: (Batch element 2): The factorization could not be completed because the input is not positive-definite (the leading minor of order 2 is not positive-definite).

As can be seen above, the basic implementation of the training loop works, decreasing the loss significantly over the epochs. Obviously this is just on a very small subset of the data so we expect significant overfitting to take place here.